In [2]:
import sys
sys.path.append("C:/Users/gruau/OneDrive/Documents/CentraleSupelec/3A/Stage Oxford/Implementation/NeoBERT/NeoBERT_dev/src/")
import importlib
import neobert.model as mdl
import neobert.tokenizer as tkn
importlib.reload(mdl)
import torch
import pandas as pd
from collections import defaultdict
#importlib.reload(mdl)

c:\Users\gruau\OneDrive\Documents\CentraleSupelec\3A\Stage Oxford\Implementation\NeoBERT\NeoBERTenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
a = 'dhfhf/eee

eee


In [3]:
from omegaconf import DictConfig
from torch.nn import CrossEntropyLoss

#A test  instantiation of the model

In [5]:
import neobert.model as mdl  # or your actual import if different

config = mdl.NeoBERTConfig(        # must be > max index in dummy_input
    hidden_size=24,             # divisible by num_attention_heads
    num_attention_heads=12,     # divides hidden_size evenly
    num_hidden_layers=12,       # small number of layers
    intermediate_size=3072,      # FFN dim
    max_length=128,             # match seq_len
    pad_token_id=0,            # define padding token index
    rope=False, 
    vocab_size = 30522,        # must be > max index in dummy_input
    expert_sizes = '0, 252, 308, 364, 420, 476, 532, 588',              # simpler
    flash_attention=False      # simpler
)

# Instantiate model
model = mdl.NeoBERTLMHead(config, )

# Generate dummy input (ensure all token indices < vocab_size)
batch_size = 100
seq_length = 128
num_experts = 8

# Token indices from 1 to vocab_size-1 (reserve 0 for pad_token)
dummy_input = torch.randint(1, config.vocab_size, (batch_size, seq_length))

# Add some padding tokens at the end (optional)
dummy_input[:, -2:] = config.pad_token_id  # last 2 tokens are padding

# Apply MLM masking with MASK token 103
mask_token_id = 103
mask_prob = 0.15  # Standard MLM masking probability

# Create random mask for MLM
mask_indices = torch.rand(batch_size, seq_length) < mask_prob
# Don't mask padding tokens
mask_indices = mask_indices & (dummy_input != config.pad_token_id)

# Apply masking
dummy_input[mask_indices] = mask_token_id

# Generate additive pad mask: 0.0 for valid tokens, -inf for padding
pad_mask = (dummy_input != config.pad_token_id).float()
pad_mask = torch.where(pad_mask == 1, 0.0, float('-inf'))

# Run forward pass
model.eval()
with torch.no_grad():
    output = model(dummy_input, pad_mask=pad_mask, output_expert_usage_loss=True, output_router_logits=True)

# Check outputs
print("Logits shape:", output["logits"].shape)
print("Expert usage loss:", output["expert_usage_loss"])



Logits shape: torch.Size([100, 128, 30522])
Expert usage loss: tensor(0.2295)


#Test  entropy per sequence

In [24]:
def get_entropy_per_sequence(gate_logits: tuple[torch.Tensor],attention_mask: torch.Tensor, entropy_loss_epsilon: float = 1e-5) -> torch.Tensor:
    """
    Computes the entropy loss per sequence for the Heterogeneous Mixture of Experts (HMoE) model.

    Args:
        gate_logits: Logits from the gate, should be a tensor of shape [batch_size, sequence_length, num_experts].
        attention_mask: The additive attention mask for the input sequences (0 for tokens to attend to, -inf for padding tokens): shape [batch_size, sequence_length].
        entropy_loss_epsilon: A small constant to avoid log(0).

    Returns:
        The average entropy  per sequence.
    """
    assert isinstance(gate_logits, tuple), "gate_logits should be a tuple of tensors, one per layer"

    n_layers = len(gate_logits)
    num_experts = gate_logits[0].shape[-1]
    batch_size, seq_length = attention_mask.shape
    compute_device = gate_logits[0].device

    #convert tuple to a single tensor
    concatenated_gate_logits = torch.cat([layer_gate.to(compute_device) for layer_gate in gate_logits], dim=0)#n_layers*batch_size*seq_length,num_experts    
    
    # Apply attention mask to ignore padding tokens
    concatenated_gate_logits = concatenated_gate_logits.reshape(n_layers,batch_size,seq_length,num_experts)#shape (n_layers,batch_size, seq_length,num_experts)
    if attention_mask is not None:
        multiplicative_attention_mask = (attention_mask != float('-inf')).float().to(compute_device) #1 for unmasked tokens, 0 for masked tokens
        concatenated_gate_logits = concatenated_gate_logits * multiplicative_attention_mask.reshape(1,batch_size,seq_length, 1) #shape (n_layers,batch_size,seq_length,num_experts)
    print(concatenated_gate_logits.shape)

    #aggregate the logits per sequence at at each layer (reason for doing this per layer: maybe at different layers a sequence is routed to a different subset of experts):
    concatenated_gate_logits = concatenated_gate_logits.reshape(-1,seq_length,num_experts)#shape (n_layers*batch_size, seq_length,num_experts)
    sequence_logits = torch.sum(concatenated_gate_logits, dim=1) #shape (n_layers*batch_size, num_experts)
    sequence_weights = torch.nn.functional.softmax(sequence_logits, dim=-1)#shape (n_layers*batch_size, num_experts)

    # Compute the entropy of the distibution of each sequence at each layer
    entropy_per_sequence = -torch.sum(sequence_weights * torch.log(sequence_weights + entropy_loss_epsilon), dim=-1) #shape (n_layers*batch_size)
    print(len(entropy_per_sequence))

    #compute the average entropy across all sequences and layers
    mean_entropy_per_sequence = torch.mean(entropy_per_sequence)
    return mean_entropy_per_sequence

In [25]:
mean_entropy_per_sequence = get_entropy_per_sequence(output["router_logits"], pad_mask)
print(mean_entropy_per_sequence)


torch.Size([12, 2, 16, 8])
24
tensor(2.0562)


#Test intra sequence cost variance

In [ ]:
def get_intra_sequence_normalised_expert_cost_variance(router_logits: torch.Tensor, attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
    """_summary_
    Compute the variance of the expert usage cost per sequence in a batch. 
    A high variance means that different tokens in the same sequence are routed to experts of different sizes.
    A low variance means that all tokens in the same sequence are routed to experts of similar sizes.

    Args:
        router_logits (Tensor): Logits from the gate, should be a tuple of n_layers tensors of shape [batch_size, sequence_length, num_experts].
        attention_mask (Tensor): The additive attention mask for the input sequences (0 for tokens to attend to, -inf for padding tokens): shape [batch_size, sequence_length]

    Returns: 
    """
    

    # Correct stacking of tuple of raw_routing_weights            
    concatenated_router_logits = torch.stack(list(router_logits), dim=0) #shape (n_layers, batch * sequence_length, n_experts)
    raw_routing_weights = torch.softmax(concatenated_router_logits, dim=-1)

    n_layers,_,n_experts = raw_routing_weights.shape
    batch_size, seq_length = attention_mask.shape

    if 2 > 1:

        # Compute the expert costs based on their sizes
        expert_dims = [int(size) for size in cfg.expert_sizes.split(',')]
        expert_dims_tensor = torch.tensor(expert_dims,dtype=torch.float32,
            device=raw_routing_weights.device )
        normalisation_factor = torch.sum(expert_dims_tensor)
        normalised_expert_sizes = expert_dims_tensor / normalisation_factor#prevent overflow
        normalised_routing_costs = normalised_expert_sizes**2
        normalised_routing_costs = normalised_routing_costs.to(dtype = raw_routing_weights.dtype)#cast back to original dtype

        #reshape  raw_routing_weights to (n_layers,batch_size,seq_length,n_experts) to then apply attention mask
        raw_routing_weights = raw_routing_weights.reshape(n_layers, batch_size, seq_length, n_experts)
        number_of_tokens_per_seq = seq_length
        

        #ignore padding_tokens
        if attention_mask!= None: #attention_mask has shape (Batch,seq_length) and routing_weights has shape (n_layers, batch * sequence_length, n_experts)
            multiplicative_mask = torch.where(attention_mask == 0, 1.0, 0.0).to(dtype = raw_routing_weights.dtype).reshape(1, batch_size, seq_length, 1)
            number_of_tokens_per_seq = multiplicative_mask.sum(dim=2).squeeze()
            raw_routing_weights = raw_routing_weights * multiplicative_mask
        # Compute usage loss per token
        normalised_expert_usage_loss = torch.einsum('mijk,k->mij', raw_routing_weights, normalised_routing_costs) #(n_layers,batch_size,seq_length)
        
        #sum across layers
        normalised_expert_usage_loss = torch.sum(normalised_expert_usage_loss, dim=0) #shape (batch_size,seq_length)

        # Compute variance across tokens in the same sequence
        mean_normalised_expert_usage_loss_per_sequence = torch.sum(normalised_expert_usage_loss, dim = 1) / number_of_tokens_per_seq #shape (batch_size,)
        mean_squared_normalised_expert_usage_loss_per_sequence = torch.sum(normalised_expert_usage_loss**2, dim = 1) / number_of_tokens_per_seq #shape (batch_size,)
        intra_sequence_normalised_expert_usage_loss_var = mean_squared_normalised_expert_usage_loss_per_sequence - mean_normalised_expert_usage_loss_per_sequence**2 #shape (batch_size,)

        # Average over sequences
        mean_intra_sequence_normalised_expert_usage_loss_var = torch.mean(intra_sequence_normalised_expert_usage_loss_var)

    else:
        mean_intra_sequence_normalised_expert_usage_loss_var = torch.tensor(0.0, device=raw_routing_weights.device)

    return mean_intra_sequence_normalised_expert_usage_loss_var #shape (1,)

In [70]:
get_intra_sequence_normalised_expert_cost_variance(output["router_logits"], None, config)

AttributeError: 'NoneType' object has no attribute 'shape'

test  expert usage logits

In [92]:
def get_expert_usage_cum_prob(gate_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
    """
    Compute the distribution of routing weights sent to each expert across all tokens in the batch and layers.
    Thhis is in absolute value, so needs to be normalised to get proportions.

    Args:
        gate_logits (tuple[torch.Tensor]): Logits from the gate, should be a tuple of n_layers tensors of shape [batch_size*sequence_length, num_experts].
        attention_mask (torch.Tensor): The additive attention mask for the input sequences (0 for tokens to attend to, -inf for padding tokens): shape [batch_size, sequence_length].

    Returns:
        torch.Tensor: The distribution of logits sent to each expert across all tokens in the batch.
    """
    assert isinstance(gate_logits, tuple), "gate_logits should be a tuple of tensors, one per layer"

    n_layers = len(gate_logits)
    num_experts = gate_logits[0].shape[-1]
    batch_size = 10000
    seq_length = 16
    compute_device = gate_logits[0].device

    # Convert tuple to a single tensor
    concatenated_gate_logits = torch.cat([layer_gate.to(compute_device) for layer_gate in gate_logits], dim=0)  # n_layers*batch_size*seq_length,num_experts

    # Apply attention mask to ignore padding tokens
    concatenated_gate_logits = concatenated_gate_logits.reshape(n_layers, batch_size, seq_length, num_experts)  # shape (n_layers,batch_size, seq_length,num_experts)
    if attention_mask is not None:
        multiplicative_attention_mask = (attention_mask != float('-inf')).float().to(compute_device)  # 1 for unmasked tokens, 0 for masked tokens
        concatenated_gate_logits = concatenated_gate_logits * multiplicative_attention_mask.reshape(1, batch_size, seq_length, 1)  # shape (n_layers,batch_size,seq_length,num_experts)

    # Aggregate the logits per expert across all tokens

    concatenated_gate_logits = concatenated_gate_logits.reshape(-1, seq_length, num_experts)  # shape (n_layers*batch_size, seq_length,num_experts)
    concatenated_gate_softmax = torch.nn.functional.softmax(concatenated_gate_logits, dim=-1)  # shape (n_layers*batch_size, seq_length,num_experts)
    expert_cum_prob = torch.sum(concatenated_gate_softmax, dim=(0,1))  # shape (n_layers*batch_size, num_experts)

    return expert_cum_prob

In [93]:
print(output["router_logits"][0])
logits = get_expert_usage_logits(output["router_logits"], pad_mask, config)
print((logits))
proportions = torch.nn.functional.softmax(logits, dim=-1)
print(proportions)


tensor([[-0.0394, -0.0222, -0.0480,  ..., -0.0211,  0.0055, -0.0970],
        [ 0.1364, -0.0334,  0.0018,  ..., -0.0516, -0.1081, -0.0403],
        [-0.0158,  0.0907, -0.0672,  ...,  0.1013,  0.0889, -0.0397],
        ...,
        [-0.1185, -0.0247, -0.0153,  ...,  0.0267, -0.0483,  0.0602],
        [-0.0669,  0.0207, -0.1129,  ...,  0.0757, -0.0268, -0.0813],
        [-0.0669,  0.0207, -0.1129,  ...,  0.0757, -0.0268, -0.0813]])
tensor([240629.7188, 238778.7969, 241068.6250, 240181.8125, 240587.7500,
        239139.8125, 240501.9375, 239111.5000])
tensor([0., 0., 1., 0., 0., 0., 0., 0.])


In [ ]:
def get_mean_expert_usage_per_layer(gate_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
    """
    Compute the mean expert selected at each layer averaged across all tokens in the batch.
    This tells us whether different layers tend to use different experts on average.

    Args:
        gate_logits (tuple[torch.Tensor]): Logits from the gate, should be a tuple of n_layers tensors of shape [batch_size*sequence_length, num_experts].
        attention_mask (torch.Tensor): The additive attention mask for the input sequences (0 for tokens to attend to, -inf for padding tokens): shape [batch_size, sequence_length].

    Returns:
        torch.Tensor: The distribution of logits sent to each expert across all tokens in the batch.
    """
    assert isinstance(gate_logits, tuple), "gate_logits should be a tuple of tensors, one per layer"

    assert isinstance(gate_logits, tuple), "gate_logits should be a tuple of tensors, one per layer"

    n_layers = len(gate_logits)
    num_experts = gate_logits[0].shape[-1]
    batch_size = 10000
    seq_length = 16
    compute_device = gate_logits[0].device

    # Convert tuple to a single tensor
    concatenated_gate_logits = torch.cat([layer_gate.to(compute_device) for layer_gate in gate_logits], dim=0)  # n_layers*batch_size*seq_length,num_experts

    # Apply attention mask to ignore padding tokens
    concatenated_gate_logits = concatenated_gate_logits.reshape(n_layers, batch_size, seq_length, num_experts)  # shape (n_layers,batch_size, seq_length,num_experts)
    if attention_mask is not None:
        multiplicative_attention_mask = (attention_mask != float('-inf')).float().to(compute_device)  # 1 for unmasked tokens, 0 for masked tokens
        concatenated_gate_logits = concatenated_gate_logits * multiplicative_attention_mask.reshape(1, batch_size, seq_length, 1)  # shape (n_layers,batch_size,seq_length,num_experts)

    # Aggregate the logits per expert across all tokens

    concatenated_gate_logits = concatenated_gate_logits.reshape(-1, seq_length, num_experts)  # shape (n_layers*batch_size, seq_length,num_experts)
    concatenated_gate_softmax = torch.nn.functional.softmax(concatenated_gate_logits, dim=-1)  # shape (n_layers*batch_size, seq_length,num_experts)
    expert_cum_prob = torch.sum(concatenated_gate_softmax, dim=(0,1))  # shape (num_experts,)

In [ ]:
Compute the distribution of routing weights sent to each expert across all tokens in the batch and layers.
    This tells us  how much each expert is used in absolute terms and we can plot this throughout training
    This is in absolute value, so needs to be normalised to get proportions.

    Args:
        gate_logits (tuple[torch.Tensor]): Logits from the gate, should be a tuple of n_layers tensors of shape [batch_size*sequence_length, num_experts].
        attention_mask (torch.Tensor): The additive attention mask for the input sequences (0 for tokens to attend to, -inf for padding tokens): shape [batch_size, sequence_length].

    Returns:
        torch.Tensor: The distribution of cumulative routing weights sent to each expert across all tokens in the batch.

In [ ]:
gate_logits = (torch.tensor([[0.1, 0.2, 0.3], [0.8, 0.2, 0.5]]), torch.tensor([[0.5, 0.5, 0.6], [0.6, 0.4, 0.3]]))
# batch_size * seq length = 2
# n_experts = 3

In [ ]:
concatenated_gate_logits = torch.cat([layer_gate for layer_gate in gate_logits], dim=0)
#nlayers*batch_zize*seq_length

In [ ]:


#token level metrics
#token_expert_usage_dict = defaultdict(lambda:[])

def get_df_of_average_token_usage(token_expert_usage_dict: defaultdict) -> pd.DataFrame:
    """Convert a dictionary of token expert usage information into a pandas DataFrame.
    """
    token_expert_usage_dict_avg = {token_id: sum(usages) / len(usages) for token_id, usages in token_expert_usage_dict.items()} 
    token_expert_usage_dict_rel_std = {token_id: torch.sqrt(torch.sum((usages - token_expert_usage_dict_avg[token_id]) ** 2)/len(usages))/ token_expert_usage_dict_avg[token_id] for token_id, usages in token_expert_usage_dict.items()}  #sort by average expert usage
    #convert this  to a panda  data frame with columns token_id, avg_expert_usage, rel_std_expert_usage
    df = pd.DataFrame({
        'token_id': list(token_expert_usage_dict_avg.keys()),
        'avg_expert_usage': list(token_expert_usage_dict_avg.values()),
        'rel_std_expert_usage': list(token_expert_usage_dict_rel_std.values())
    })
    return df


def _append_expert_usage_per_token(token_expert_usage_dict: defaultdict, batch_input_id:torch.Tensor, batch_gate_logits: tuple[torch.Tensor], batch_attention_mask: torch.Tensor, cfg: DictConfig) -> defaultdict:
    """Append expert usage information for each token in the batch.
    This function builds a given dictionary where the key is the token_id and the value is a list of expert usage values for that token across different occurrences in the batch.
    Expert usage is defined as the mean expert index selected across all layers for a given token. In the end we really care about how routing is going on, not the expert loss
    (1)This can be used to verify whether at a given training points, a given token follows a consistant routing pattern.
    (2)Taking averages across all occurrences of a token, it can help rank tokens by complexity
    (3)Evolution of token complexity throughout  training: if the relative diff between token complexity and mean complexity diminishes with number of occurrences, then model 
    learns to make token simpler as it sees  it more often. This is similar to correlating mse with expert complexity

    Args:
        token_expert_usage_dict (defaultdict(lambda:[]): Dictionary to store expert usage information (key: token_id, value: list of expert usage values).
        batch_input_id (torch.Tensor): Input IDs for the batch.
        batch_gate_logits (tuple[torch.Tensor]): Gate logits for the batch.
    """
    n_layers = len(batch_gate_logits)
    num_experts = batch_gate_logits[0].shape[-1]
    batch_size = cfg.dataloader.train.batch_size
    seq_length = cfg.tokenizer.max_length
    compute_device = batch_gate_logits[0].device
    expert_indices = torch.arange(num_experts, device=compute_device, dtype=batch_gate_logits[0].dtype)

    batch_concatenated_gate_logits = _get_concatenated_gate_logits(batch_gate_logits, batch_attention_mask, cfg)#shape (n_layers,batch_size, seq_length,num_experts)
    batch_concatenated_gate_softmax = torch.nn.functional.softmax(batch_concatenated_gate_logits, dim=-1) #shape (n_layers,batch_size, seq_length,num_experts)
    mean_selected_expert_per_token =  torch.einsum('nbse,e->nbs', batch_concatenated_gate_softmax, expert_indices)# shape (n_layers,batch_size, seq_length))
    mean_expert_usage_per_token = torch.mean(mean_selected_expert_per_token, dim=0)  # shape (batch_size, seq_length): mean across layers
    mean_expert_usage_per_token = mean_expert_usage_per_token.reshape(-1)  # shape (batch_size*seq_length,)
    
    for token_index in range(batch_size * seq_length):
        token_id = batch_input_id.reshape(-1)[token_index].item() #why this?
        expert_usage = mean_expert_usage_per_token[token_index].item()
        token_expert_usage_dict[token_id].append(expert_usage)
    
    return token_expert_usage_dict


In [130]:
def get_correlation_mean_expert_usage_across_layers(mean_expert_usage_per_layer: torch.Tensor) -> torch.Tensor:
    """Compute the correlation between the mean expert between layer N and layer N+1 for a given token across all tokens and all N,N+1 pairs.

    Args:
        mean_expert_usage_per_layer (torch.Tensor): shape [n_layers,n_seq_length*batch_size*number_of_batches]
    """
    n_layers = mean_expert_usage_per_layer.shape[0]
    correlations = []
    for i in range(n_layers - 1):
        layer_n = mean_expert_usage_per_layer[i]
        layer_n_plus_1 = mean_expert_usage_per_layer[i + 1]
        print(f"Layer {i} std: {torch.std(layer_n):.4f}, Layer {i+1} std: {torch.std(layer_n_plus_1):.4f}")
        if torch.std(layer_n) > 0 and torch.std(layer_n_plus_1) > 0:
            correlation = torch.corrcoef(torch.stack([layer_n, layer_n_plus_1]))[0, 1]
            correlations.append(correlation)
    if len(correlations) > 0:
        mean_correlation = torch.mean(torch.stack(correlations))
    else:
        mean_correlation = torch.tensor(0.0, device=mean_expert_usage_per_layer.device)
    return mean_correlation


def get_mean_expert_usage_per_layer(gate_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
    """
    Compute the mean expert selected at each layer averaged across all tokens in the batch.
    This tells us whether different layers tend to use different experts on average.

    Args:
        gate_logits (tuple[torch.Tensor]): Logits from the gate, should be a tuple of n_layers tensors of shape [batch_size*sequence_length, num_experts].
        attention_mask (torch.Tensor): The additive attention mask for the input sequences (0 for tokens to attend to, -inf for padding tokens): shape [batch_size, sequence_length].

    Returns:
        torch.Tensor: The distribution of logits sent to each expert across all tokens in the batch.
    """
    mean_selected_expert_per_layer_per_token = _get_mean_expert_usage_per_layer_per_token(gate_logits, attention_mask, cfg)  # shape (n_layers,batch_size*seq_length)
    mean_selected_expert_per_layer = torch.mean(mean_selected_expert_per_layer_per_token, dim=1)  # shape (n_layers,) #mean across tokens
    return mean_selected_expert_per_layer



def _get_mean_expert_usage_per_layer_per_token(gate_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
    assert isinstance(gate_logits, tuple), "gate_logits should be a tuple of tensors, one per layer"

    n_layers = len(gate_logits)
    num_experts = gate_logits[0].shape[-1]
    compute_device = gate_logits[0].device
    expert_indices = torch.arange(num_experts, device=compute_device, dtype=gate_logits[0].dtype)

    # Convert tuple to a single tensor
    concatenated_gate_logits = _get_concatenated_gate_logits(gate_logits, attention_mask, cfg)  # n_layers*batch_size*seq_length,num_experts

    # Aggregate the probabilities per expert across all tokens
    concatenated_gate_logits_per_layer = concatenated_gate_logits.reshape(n_layers,-1, num_experts)  # shape (n_layers,batch_size*seq_length,num_experts)
    concatenated_gate_softmax_per_layer = torch.nn.functional.softmax(concatenated_gate_logits_per_layer, dim=-1) # shape (n_layers,batch_size*seq_length,num_experts)
    mean_selected_expert_per_layer = torch.einsum('men,n->me', concatenated_gate_softmax_per_layer, expert_indices)  # shape (n_layers,batch_size*seq_length)

    return mean_selected_expert_per_layer


def _get_concatenated_gate_logits(gate_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
    """
    Concatenate the gate logits from all layers into a single tensor.

    Args:
        gate_logits (tuple[torch.Tensor]): Logits from the gate, should be a tuple of n_layers tensors of shape [batch_size*sequence_length, num_experts].
        attention_mask (torch.Tensor): The additive attention mask for the input sequences (0 for tokens to attend to, -inf for padding tokens): shape [batch_size, sequence_length].

    Returns:
        torch.Tensor: Concatenated gate logits of shape [n_layers*batch_size*sequence_length, num_experts].
    """
    assert isinstance(gate_logits, tuple), "gate_logits should be a tuple of tensors, one per layer"

    n_layers = len(gate_logits)
    compute_device = gate_logits[0].device

    # Convert tuple to a single tensor
    concatenated_gate_logits = torch.cat([layer_gate.to(compute_device) for layer_gate in gate_logits], dim=0)  # n_layers*batch_size*seq_length,num_experts
    # Apply attention mask to ignore padding tokens
    concatenated_gate_logits = concatenated_gate_logits.reshape(n_layers, batch_size, seq_length, -1)  # shape (n_layers,batch_size, seq_length,num_experts)
    if attention_mask is not None:
        multiplicative_attention_mask = (attention_mask != float('-inf')).float().to(compute_device)  # 1 for unmasked tokens, 0 for masked tokens
        concatenated_gate_logits = concatenated_gate_logits * multiplicative_attention_mask.reshape(1, batch_size, seq_length, 1)  # shape (n_layers,batch_size,seq_length,num_experts)

    return concatenated_gate_logits

In [160]:
mean_expert_usage_per_layer = get_mean_expert_usage_per_layer(output["router_logits"], pad_mask, config)
print(mean_expert_usage_per_layer)

tensor([3.5176, 3.4919, 3.5131, 3.5048, 3.5053, 3.5046, 3.5054, 3.5088, 3.5059,
        3.4932, 3.4865, 3.4897])


In [156]:
mean_expert_usage_per_layer_per_token = _get_mean_expert_usage_per_layer_per_token(output["router_logits"], pad_mask, config)
print(mean_expert_usage_per_layer_per_token)

tensor([[3.4954, 3.5519, 3.5310,  ..., 3.5419, 3.5000, 3.5000],
        [3.3991, 3.5308, 3.3908,  ..., 3.4656, 3.5000, 3.5000],
        [3.4900, 3.5823, 3.5587,  ..., 3.5463, 3.5000, 3.5000],
        ...,
        [3.4780, 3.4655, 3.4645,  ..., 3.4438, 3.5000, 3.5000],
        [3.4535, 3.4582, 3.4453,  ..., 3.4342, 3.5000, 3.5000],
        [3.5023, 3.4596, 3.4790,  ..., 3.5374, 3.5000, 3.5000]])


In [162]:
corr = get_correlation_mean_expert_usage_across_layers(mean_expert_usage_per_layer_per_token)
print(f"corr = {corr:.2f}")

Layer 0 std: 0.0360, Layer 1 std: 0.0532
Layer 1 std: 0.0532, Layer 2 std: 0.0398
Layer 2 std: 0.0398, Layer 3 std: 0.0385
Layer 3 std: 0.0385, Layer 4 std: 0.0447
Layer 4 std: 0.0447, Layer 5 std: 0.0318
Layer 5 std: 0.0318, Layer 6 std: 0.0360
Layer 6 std: 0.0360, Layer 7 std: 0.0365
Layer 7 std: 0.0365, Layer 8 std: 0.0303
Layer 8 std: 0.0303, Layer 9 std: 0.0431
Layer 9 std: 0.0431, Layer 10 std: 0.0342
Layer 10 std: 0.0342, Layer 11 std: 0.0379
corr = 0.03


In [176]:
def get_df_of_average_token_usage(token_expert_usage_dict: defaultdict) -> pd.DataFrame:
    """Convert a dictionary of token expert usage information into a pandas DataFrame.
    """
    token_expert_usage_dict_avg = {token_id: sum(usages) / len(usages) for token_id, usages in token_expert_usage_dict.items()} 
    token_expert_usage_dict_rel_std = {token_id: np.sqrt(np.sum((np.array(usages) - token_expert_usage_dict_avg[token_id]) ** 2)/len(usages))/ token_expert_usage_dict_avg[token_id] for token_id, usages in token_expert_usage_dict.items()}  #sort by average expert usage
    #convert this  to a panda  data frame with columns token_id, avg_expert_usage, rel_std_expert_usage
    df = pd.DataFrame({
        'token_id': list(token_expert_usage_dict_avg.keys()),
        'avg_expert_usage': list(token_expert_usage_dict_avg.values()),
        'rel_std_expert_usage': list(token_expert_usage_dict_rel_std.values())
    })
    return df


def _append_expert_usage_per_token(token_expert_usage_dict: defaultdict, batch_input_id:torch.Tensor, batch_gate_logits: tuple[torch.Tensor], batch_attention_mask: torch.Tensor, cfg: DictConfig) -> defaultdict:
    """Append expert usage information for each token in the batch.
    This function builds a given dictionary where the key is the token_id and the value is a list of expert usage values for that token across different occurrences in the batch.
    Expert usage is defined as the mean expert index selected across all layers for a given token. In the end we really care about how routing is going on, not the expert loss
    (1)This can be used to verify whether at a given training points, a given token follows a consistant routing pattern.
    (2)Taking averages across all occurrences of a token, it can help rank tokens by complexity
    (3)Evolution of token complexity throughout  training: if the relative diff between token complexity and mean complexity diminishes with number of occurrences, then model 
    learns to make token simpler as it sees  it more often. This is similar to correlating mse with expert complexity

    Args:
        token_expert_usage_dict (defaultdict(lambda:[]): Dictionary to store expert usage information (key: token_id, value: list of expert usage values).
        batch_input_id (torch.Tensor): Input IDs for the batch.
        batch_gate_logits (tuple[torch.Tensor]): Gate logits for the batch.
    """
    n_layers = len(batch_gate_logits)
    num_experts = batch_gate_logits[0].shape[-1]
    compute_device = batch_gate_logits[0].device
    expert_indices = torch.arange(num_experts, device=compute_device, dtype=batch_gate_logits[0].dtype)

    batch_concatenated_gate_logits = _get_concatenated_gate_logits(batch_gate_logits, batch_attention_mask, cfg)#shape (n_layers,batch_size, seq_length,num_experts)
    batch_concatenated_gate_softmax = torch.nn.functional.softmax(batch_concatenated_gate_logits, dim=-1) #shape (n_layers,batch_size, seq_length,num_experts)
    mean_selected_expert_per_token =  torch.einsum('nbse,e->nbs', batch_concatenated_gate_softmax, expert_indices)# shape (n_layers,batch_size, seq_length))
    mean_expert_usage_per_token = torch.mean(mean_selected_expert_per_token, dim=0)  # shape (batch_size, seq_length): mean across layers
    mean_expert_usage_per_token = mean_expert_usage_per_token.reshape(-1)  # shape (batch_size*seq_length,)
    
    for token_index in range(batch_size * seq_length):
        token_id = batch_input_id.reshape(-1)[token_index].item()
        expert_usage = mean_expert_usage_per_token[token_index].item()
        token_expert_usage_dict[token_id].append(expert_usage)
    
    return token_expert_usage_dict

In [195]:
dico = defaultdict(lambda:[])
dummy_input_id = torch.randint(1, config.vocab_size, (batch_size, seq_length))
dico = _append_expert_usage_per_token(dico, dummy_input_id, output["router_logits"], pad_mask, config)
print(dico)
print(len(dico))
df = get_df_of_average_token_usage(dico)
print(df)
print(type(dico[17][0]))

defaultdict(<function <lambda> at 0x00000273679AF7E0>, {3561: [3.515974760055542], 14656: [3.508420944213867], 18901: [3.4922897815704346], 28114: [3.47885799407959, 3.510941505432129], 23203: [3.487917900085449], 15798: [3.4907782077789307], 6751: [3.4998083114624023], 16842: [3.495382070541382], 12378: [3.5174038410186768], 27722: [3.5137290954589844, 3.5101568698883057], 12395: [3.485595464706421], 19524: [3.4778010845184326], 24699: [3.5081489086151123], 4196: [3.490341901779175], 2598: [3.5], 12595: [3.5], 23135: [3.527219533920288], 14650: [3.503797769546509], 12687: [3.503061056137085], 502: [3.4821062088012695], 10255: [3.4927823543548584], 10585: [3.5155107975006104], 1894: [3.4891176223754883], 29825: [3.5035102367401123], 18602: [3.5245440006256104], 21238: [3.4824697971343994], 3207: [3.4839420318603516], 3989: [3.4832382202148438], 27564: [3.5024473667144775], 15882: [3.4995107650756836], 7703: [3.5], 4982: [3.5], 20235: [3.5034987926483154], 2504: [3.5062544345855713], 21

IndexError: list index out of range

In [190]:
def rewrite_inputs_with_tokenizer(input_ids,attention_mask, tokenizer):
    """
    Takes tokenized inputs and rewrites/detokenizes them back to text using the tokenizer.
    
    Args:
        batch: Dictionary containing 'input_ids' and optionally 'attention_mask'
        tokenizer: The tokenizer used for encoding/decoding
    
    Returns:
        Dictionary with original inputs and rewritten text
    """
    results = []
    
    for i in range(input_ids.shape[0]):
        # Get individual sequence
        sequence_ids = input_ids[i]
        
        # If attention mask exists, use it to identify valid tokens
        if attention_mask is not None:
            # Convert additive mask back to standard mask for processing
            if attention_mask.dtype == torch.float32:
                # Additive mask: 0.0 for valid tokens, -inf for padding
                valid_mask = (attention_mask[i] != float('-inf'))
            else:
                # Standard mask: 1 for valid tokens, 0 for padding
                valid_mask = (attention_mask[i] == 1)
            
            # Only keep valid tokens
            valid_ids = sequence_ids[valid_mask]
        else:
            valid_ids = sequence_ids
        
        # Decode the sequence back to text
        decoded_text = tokenizer.decode(valid_ids, skip_special_tokens=True)
        
        # Store results
        results.append({
            'original_ids': sequence_ids.tolist(),
            'valid_ids': valid_ids.tolist(),
            'rewritten_text': decoded_text
        })
    
    return results



In [8]:
tokenizer = tkn.get_tokenizer(pretrained_model_name_or_path ="google-bert/bert-base-uncased", trust_remote_code = True, max_length = seq_length, vocab_size = config.vocab_size)
# Example usage with the existing batch
print("Testing rewrite function with STSB tokenizer:")
example_results = rewrite_inputs_with_tokenizer(dummy_input_id, pad_mask,tokenizer)

for i, result in enumerate(example_results):
    print(f"\nSample {i+1}:")
    print(f"Original token IDs: {result['original_ids'][:20]}...")  # Show first 20 tokens
    print(f"Valid token IDs: {result['valid_ids'][:20]}...")
    print(f"Rewritten text: {result['rewritten_text']}")

Testing rewrite function with STSB tokenizer:


NameError: name 'rewrite_inputs_with_tokenizer' is not defined

In [200]:
rewritten = [tokenizer.decode(list(token_id), skip_special_tokens=True) for token_id in dummy_input_id[0:5]]
print(rewritten)

['filledtical cocktailzumi ogden luckily arkansas niger handball administrations mkendra revisions lifted ground tasted', 'nightstandhaven breakthrough [unused497] sends civilization 氵شmler supervisingdeization hugely wallet incumbent grow', 'bandwidth level name [unused938] appeal baton austria ramps dooiegδ jozef younger mcpherson labor granddaughter', 'latecisewas fernando copperchu torpedo randolph lithuania speedsawacek 明 react stagecoach difficulty', 'torahzumi म cruises shortest レ ɑ measure footingntly [unused750]lde 1870s havoc pedestalstatic']


In [10]:
def get_inter_token_normalised_expert_cost_rel_std(router_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
    """Compute the relative standard deviation of the expert usage cost across all tokens in a batch: std/mean"""
    
    normalised_expert_usage_loss_per_token_and_layer, number_of_tokens_per_seq = _get_normalised_expert_usage_per_token_and_layer(router_logits, attention_mask, cfg)#(n_layers,batch_size,seq_length)
    num_tokens = torch.sum(number_of_tokens_per_seq).item()
    normalised_expert_usage_loss_per_token = torch.sum(normalised_expert_usage_loss_per_token_and_layer, dim = 0).reshape(-1)#shape (seq_length*batch_size)

    mean_normalised_expert_usage_loss = torch.sum(normalised_expert_usage_loss_per_token)/ num_tokens #shape (1,)
    print("mean inter token:",mean_normalised_expert_usage_loss)
    var_normalised_expert_usage_loss = torch.sum((normalised_expert_usage_loss_per_token - mean_normalised_expert_usage_loss)**2) / num_tokens #shape (1,)

    inter_token_normalised_expert_usage_loss_rel_std = torch.sqrt(var_normalised_expert_usage_loss) / (mean_normalised_expert_usage_loss + 1e-10) #shape (1,): relative std = std/mean
    return inter_token_normalised_expert_usage_loss_rel_std #shape (1,)

def get_inter_sequence_normalised_expert_cost_rel_std( router_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
    """_summary_
    Compute the relative standard deviation of the expert usage cost across all sequences in a batch: std/mean
    
    Args:
        router_logits (Tensor): Logits from the gate, should be a tuple of n_layers tensors of shape [batch_size*sequence_length, num_experts].
        attention_mask (Tensor): The additive attention mask for the input sequences (0 for tokens to attend to, -inf for padding tokens): shape [batch_size, sequence_length]

    Returns: 
    """
    normalised_mean_expert_usage_loss_per_sequence = _get_normalised_expert_usage_cost_per_sequence(router_logits, attention_mask, cfg) #shape (batch_size,)
    # Compute RSD across sequences in the batch
    mean_normalised_expert_usage_loss = torch.mean(normalised_mean_expert_usage_loss_per_sequence) #shape (1,)
    print("mean inter sequence:",mean_normalised_expert_usage_loss)
    inter_sequence_normalised_expert_usage_loss_var = torch.var(normalised_mean_expert_usage_loss_per_sequence)
    inter_sequence_normalised_expert_usage_loss_rel_std = torch.sqrt(inter_sequence_normalised_expert_usage_loss_var) / (mean_normalised_expert_usage_loss + 1e-10) #shape (1,): relative std = std/mean
    return inter_sequence_normalised_expert_usage_loss_rel_std #shape (1,)

def get_intra_sequence_normalised_expert_cost_rel_std(router_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
    """_summary_
    Compute the relative standard deviation of the expert usage cost per sequence in a batch: std/mean
    A high standard deviation means that different tokens in the same sequence are routed to experts of different sizes. 
    This is to be compared to the same metric across all tokens in the batch
    We normalise by the mean since  std(aX) = a std(X) and otherwise the metric would be biased towards sequences with a higher expert usage cost.
    This is a similar metric to the mean entropy per sequence but it takes into account the full pathway across all layers

    Args:
        router_logits (Tensor): Logits from the gate, should be a tuple of n_layers tensors of shape [batch_size*sequence_length, num_experts].
        attention_mask (Tensor): The additive attention mask for the input sequences (0 for tokens to attend to, -inf for padding tokens): shape [batch_size, sequence_length]

    Returns: 
    """
    normalised_expert_usage_loss_per_token_and_layer, number_of_tokens_per_seq = _get_normalised_expert_usage_per_token_and_layer(router_logits, attention_mask, cfg)#(n_layers,batch_size,seq_length)
    #sum across layers
    normalised_expert_usage_loss_per_token = torch.sum(normalised_expert_usage_loss_per_token_and_layer, dim=0) #shape (batch_size,seq_length)

    # Compute RSD across tokens in the same sequence
    mean_normalised_expert_usage_loss_per_sequence = torch.sum(normalised_expert_usage_loss_per_token, dim = 1) / number_of_tokens_per_seq #shape (batch_size,)
    print("mean intra seq:",mean_normalised_expert_usage_loss_per_sequence)
    mean_squared_normalised_expert_usage_loss_per_sequence = torch.sum(normalised_expert_usage_loss_per_token**2, dim = 1) / number_of_tokens_per_seq #shape (batch_size,)
    intra_sequence_normalised_expert_usage_loss_var = mean_squared_normalised_expert_usage_loss_per_sequence - mean_normalised_expert_usage_loss_per_sequence**2 #shape (batch_size,)
    intra_sequence_normalised_expert_usage_loss_rel_std = torch.sqrt(intra_sequence_normalised_expert_usage_loss_var) / (mean_normalised_expert_usage_loss_per_sequence + 1e-10) #shape (batch_size,): relative std = std/mean

    # Average over sequences
    mean_intra_sequence_normalised_expert_usage_loss_rel_std = torch.mean(intra_sequence_normalised_expert_usage_loss_rel_std)

    return mean_intra_sequence_normalised_expert_usage_loss_rel_std #shape (1,)


def _get_normalised_expert_usage_cost_per_sequence(router_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
    """_summary_
    Compute the expert usage cost per sequence in a batch. 
    Result is normalised by the  size of the experts as we do not care about the absolute value.

    Args:
        router_logits (_type_): _description_
        attention_mask (_type_): _description_
        cfg (_type_): _description_

    Returns:
        _type_: _description_
    """
    normalised_expert_usage_loss_per_token_and_layer, number_of_tokens_per_seq = _get_normalised_expert_usage_per_token_and_layer(router_logits, attention_mask, cfg)#(n_layers,batch_size,seq_length)   
    #sum across layers
    normalised_expert_usage_loss_per_token = torch.sum(normalised_expert_usage_loss_per_token_and_layer, dim=0) #shape (batch_size,seq_length)
    # Average over all tokens in the sequence
    normalised_mean_expert_usage_loss_per_sequence = torch.sum(normalised_expert_usage_loss_per_token, dim = 1) / number_of_tokens_per_seq #shape (batch_size,)

    return normalised_mean_expert_usage_loss_per_sequence #shape (batch_size,) or None if single expert

def _get_normalised_expert_usage_per_token_and_layer(router_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, cfg: DictConfig) -> tuple[torch.Tensor,torch.Tensor]:
    # Correct stacking of tuple of raw_routing_weights            
    router_logits = torch.stack(list(router_logits), dim=0) #shape (n_layers, batch * sequence_length, n_experts)
    raw_routing_weights = torch.softmax(router_logits, dim=-1)
    n_layers,_,n_experts = raw_routing_weights.shape
    batch_size, seq_length = attention_mask.shape

    if len(cfg.expert_sizes.split(',')) > 1:

        expert_dims = [int(size) for size in cfg.expert_sizes.split(',')]
        expert_dims_tensor = torch.tensor(expert_dims,dtype=torch.float32,
            device=raw_routing_weights.device )
        
        normalisation_factor = torch.sum(expert_dims_tensor)
        normalised_expert_sizes = expert_dims_tensor / normalisation_factor#prevent overflow

        routing_costs = normalised_expert_sizes**2
        routing_costs = routing_costs.to(dtype = raw_routing_weights.dtype)#cast back to original dtype

        raw_routing_weights = raw_routing_weights.reshape(n_layers, batch_size, seq_length, n_experts)

        #ignore padding_tokens
        if attention_mask!= None: #attention_mask has shape (Batch,seq_length) and routing_weights has shape (n_layers, batch * sequence_length, n_experts)
            multiplicative_mask = torch.where(attention_mask == 0, 1.0, 0.0).to(dtype = raw_routing_weights.dtype).reshape(1, batch_size, seq_length, 1)
            number_of_tokens_per_seq = multiplicative_mask.sum(dim=2).squeeze() #shape (batch_size,)
            raw_routing_weights = raw_routing_weights * multiplicative_mask
        # Compute usage loss per token
        expert_usage_loss_per_token_and_layer = torch.einsum('mijk,k->mij', raw_routing_weights, routing_costs) #(n_layers,batch_size,seq_length)
    else: 
        expert_usage_loss_per_token_and_layer = torch.zeros((n_layers,batch_size,seq_length),device=raw_routing_weights.device)
        number_of_tokens_per_seq = torch.ones(batch_size)*seq_length
    
    return expert_usage_loss_per_token_and_layer, number_of_tokens_per_seq

In [47]:
#test all the functions in the previous cell

inter_token_rel_std = get_inter_token_normalised_expert_cost_rel_std(output["router_logits"], pad_mask, config)
inter_sequence_rel_std = get_inter_sequence_normalised_expert_cost_rel_std(output["router_logits"], pad_mask, config)#intra_sequence_rel_std = get_intra_sequence_normalised_expert_cost_rel_std(output["router_logits"], pad_mask, config)
intra_sequence_rel_std = get_intra_sequence_normalised_expert_cost_rel_std(output["router_logits"], pad_mask, config)
print(f"inter_token_rel_std = {inter_token_rel_std:.4f}")
print(f"inter_sequence_rel_std = {inter_sequence_rel_std:.4f}")
print(f"intra_sequence_rel_std = {intra_sequence_rel_std:.4f}")

mean inter token: tensor(0.2296)
mean inter sequence: tensor(0.2296)
inter_token_rel_std = 0.1261
inter_sequence_rel_std = nan
intra_sequence_rel_std = 0.0042


In [46]:
def get_intra_sequence_normalised_expert_cost_rel_std(router_logits: torch.Tensor, attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
        """_summary_
        Compute the relative standard deviation of the expert usage cost per sequence in a batch: std/mean
        A high standard deviation means that different tokens in the same sequence are routed to experts of different sizes. 
        This is to be compared to the same metric across all tokens in the batch
        We normalise by the mean since  std(aX) = a std(X) and otherwise the metric would be biased towards sequences with a higher expert usage cost.
        This is a similar metric to the mean entropy per sequence but it takes into account the full pathway across all layers

        Args:
            router_logits (Tensor): Logits from the gate, should be a tuple of n_layers tensors of shape [batch_size*sequence_length, num_experts].
            attention_mask (Tensor): The additive attention mask for the input sequences (0 for tokens to attend to, -inf for padding tokens): shape [batch_size, sequence_length]

        Returns: 
        """
    # Correct stacking of tuple of raw_routing_weights            
        concatenated_router_logits = torch.stack(list(router_logits), dim=0) #shape (n_layers, batch * sequence_length, n_experts)
        raw_routing_weights = torch.softmax(concatenated_router_logits, dim=-1)

        n_layers,_,n_experts = raw_routing_weights.shape
        batch_size = 100
        seq_length = 128

        if len(cfg.expert_sizes.split(',')) > 1:

            # Compute the expert costs based on their sizes
            expert_dims = [int(size) for size in cfg.expert_sizes.split(',')]
            expert_dims_tensor = torch.tensor(expert_dims,dtype=torch.float32,
                device=raw_routing_weights.device )
            normalisation_factor = torch.sum(expert_dims_tensor)
            normalised_expert_sizes = expert_dims_tensor / normalisation_factor#prevent overflow
            normalised_routing_costs = normalised_expert_sizes**cfg.expert_cost_exponent
            normalised_routing_costs = normalised_routing_costs.to(dtype = raw_routing_weights.dtype)#cast back to original dtype

            #reshape  raw_routing_weights to (n_layers,batch_size,seq_length,n_experts) to then apply attention mask
            raw_routing_weights = raw_routing_weights.reshape(n_layers, batch_size, seq_length, n_experts)
            number_of_tokens_per_seq = seq_length
            

            #ignore padding_tokens
            if attention_mask!= None: #attention_mask has shape (Batch,seq_length) and routing_weights has shape (n_layers, batch * sequence_length, n_experts)
                multiplicative_mask = torch.where(attention_mask == 0, 1.0, 0.0).to(dtype = raw_routing_weights.dtype).reshape(1, batch_size, seq_length, 1)
                number_of_tokens_per_seq = multiplicative_mask.sum(dim=2).squeeze()
                raw_routing_weights = raw_routing_weights * multiplicative_mask
            # Compute usage loss per token
            normalised_expert_usage_loss = torch.einsum('mijk,k->mij', raw_routing_weights, normalised_routing_costs) #(n_layers,batch_size,seq_length)
            
            #sum across layers
            normalised_expert_usage_loss = torch.sum(normalised_expert_usage_loss, dim=0) #shape (batch_size,seq_length)

            # Compute RSD across tokens in the same sequence
            mean_normalised_expert_usage_loss_per_sequence = torch.sum(normalised_expert_usage_loss, dim = 1) / number_of_tokens_per_seq #shape (batch_size,)
            mean_squared_normalised_expert_usage_loss_per_sequence = torch.sum(normalised_expert_usage_loss**2, dim = 1) / number_of_tokens_per_seq #shape (batch_size,)
            intra_sequence_normalised_expert_usage_loss_var = mean_squared_normalised_expert_usage_loss_per_sequence - mean_normalised_expert_usage_loss_per_sequence**2 #shape (batch_size,)
            intra_sequence_normalised_expert_usage_loss_rel_std = torch.sqrt(intra_sequence_normalised_expert_usage_loss_var) / (mean_normalised_expert_usage_loss_per_sequence + 1e-10) #shape (batch_size,): relative std = std/mean

            # Average over sequences
            mean_intra_sequence_normalised_expert_usage_loss_rel_std = torch.mean(intra_sequence_normalised_expert_usage_loss_rel_std)


        else:
            mean_intra_sequence_normalised_expert_usage_loss_rel_std = torch.tensor(0.0, device=raw_routing_weights.device)


        return mean_intra_sequence_normalised_expert_usage_loss_rel_std #shape (1,)

In [4]:
def get_mse_per_token(logits, cfg, batch):
        """
        Compute the Mean Squared Error (MSE) loss per token.

        Args:
            logits: The model predictions. #shape (batch_size, seq_length, vocab_size)
            cfg: Configuration object containing model parameters.
            labels: The ground truth labels. #shape (batch_size, seq_length)

        Returns:
            The computed MSE loss per token.

        """
        mlm_loss_fn = CrossEntropyLoss(reduction = "none")
        train_loss = masked_lm_loss = mlm_loss_fn(logits.view(-1, cfg.tokenizer.vocab_size), batch["labels"].view(-1))#shape seq_length*nbatch_size
        masked_lm_loss = masked_lm_loss.reshape(logits.size(0), logits.size(1))#shape (batch_size, seq_length)
        mask = (batch["labels"] != -100).float()  # (batch_size, seq_len)
        #reshape
        mask = mask.reshape(-1) #shape (batch_size*seq_length,)
        masked_lm_loss = masked_lm_loss.reshape(-1) #shape (batch_size*seq_length,)
        #extract valid tokens only
        filtered_masked_lm_loss = masked_lm_loss[mask==1]#shape (num_valid_tokens,)
    
        return filtered_masked_lm_loss #shape (num_valid_tokens,)

In [ ]:
def get_normalised_expert_usage_cost_per_token(router_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, cfg: DictConfig) -> torch.Tensor:
        """_summary_
        Compute the expert usage cost per token in a batch. 
        Result is normalised by the size of the experts as we do not care about the absolute value.

        Args:
            router_logits (_type_): _description_
            attention_mask (_type_): _description_
            cfg (_type_): _description_

        Returns:
            _type_: _description_
        """
        normalised_expert_usage_loss_per_token_and_layer, number_of_tokens_per_seq = _get_normalised_expert_usage_per_token_and_layer(router_logits, attention_mask, cfg)#(n_layers,batch_size,seq_length)   
        #sum across layers
        normalised_expert_usage_loss_per_token = torch.sum(normalised_expert_usage_loss_per_token_and_layer, dim=0) #shape (batch_size,seq_length)

        filtered_normalised_expert_usage_loss_per_token = normalised_expert_usage_loss_per_token[normalised_expert_usage_loss_per_token != 0] #remove padding tokens

        return filtered_normalised_expert_usage_loss_per_token #shape (batch_size,seq_length) or None if single expert

In [9]:
normalised_expert_usage_per_token = get_normalised_expert_usage_cost_per_token(output["router_logits"], pad_mask, config)
print(normalised_expert_usage_per_token.shape)
print(batch_size*seq_length)

torch.Size([12600])
12800


In [12]:
def get_normalised_expert_usage_cost_per_masked_token(router_logits: tuple[torch.Tensor], attention_mask: torch.Tensor, input_ids:torch.Tensor, cfg: DictConfig,tokenizer) -> torch.Tensor:
        """_summary_
        Compute the expert usage cost per token in a batch. 
        Result is normalised by the size of the experts as we do not care about the absolute value.

        Args:
            router_logits (_type_): _description_
            attention_mask (_type_): _description_
            cfg (_type_): _description_

        Returns:
            _type_: _description_
        """
        normalised_expert_usage_loss_per_token_and_layer, number_of_tokens_per_seq = _get_normalised_expert_usage_per_token_and_layer(router_logits, attention_mask, cfg)#(n_layers,batch_size,seq_length)   
        #sum across layers
        normalised_expert_usage_loss_per_token = torch.sum(normalised_expert_usage_loss_per_token_and_layer, dim=0) #shape (batch_size,seq_length)
        

        mask_token_id = tokenizer.mask_token_id #103
        padding_token_id = tokenizer.pad_token_id#0
        mask = ((input_ids == mask_token_id) & (input_ids != padding_token_id)).float()  # (batch_size, seq_len) # keep masked tokens which are not padding tokens

        #reshape 
        mask = mask.reshape(-1) #shape (batch_size*seq_length,)
        normalised_expert_usage_loss_per_token = normalised_expert_usage_loss_per_token.reshape(-1) #shape (batch_size*seq_length,)
        filtered_normalised_expert_usage_loss_per_token = normalised_expert_usage_loss_per_token[mask == 1] #keep only masked tokens

        return filtered_normalised_expert_usage_loss_per_token #shape (batch_size,seq_length) or None if single expert

In [13]:
res = get_normalised_expert_usage_cost_per_masked_token(output["router_logits"], pad_mask, dummy_input, config,tokenizer)

In [15]:
print(res.shape)
print(dummy_input.shape)

torch.Size([1877])
torch.Size([100, 128])
